In [1]:
import duckdb
import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display

### 1. Carregar dados do DuckDB

In [2]:
conn = duckdb.connect("../datalake.db")
df_raw = conn.sql("SELECT * FROM gold.expectativa_vs_realidade").df()
conn.close()

### 2. Mapeamento das opções de interface

In [3]:
opcoes_janela = {
    "1 Mês Antes": ("exp_1m_antes", "erro_1m"),
    "3 Meses Antes": ("exp_3m_antes", "erro_3m"),
    "6 Meses Antes": ("exp_6m_antes", "erro_6m"),
    "12 Meses Antes": ("exp_12m_antes", "erro_12m"),
}

### 3. Função para renderizar o gráfico

In [4]:
def plotar_grafico(janela_selecionada):
    df = df_raw[df_raw["reuniao_ano"] >= 2022].sort_values(
        "reuniao_data", ascending=True
    )

    col_exp, col_erro = opcoes_janela[janela_selecionada]

    mae = df[col_erro].abs().mean()
    bias = df[col_erro].mean()

    fig = go.Figure()

    # Linha 1: Realidade (COPOM)
    fig.add_trace(
        go.Scatter(
            x=df["reuniao_data"],
            y=df["realidade_selic"],
            mode="lines+markers",
            name="Realidade (COPOM)",
            line=dict(color="#EF553B", width=2.5),
            hovertemplate="Data: %{x}<br>Selic Real: %{y:.2f}%<extra></extra>",
        )
    )

    # Linha 2: Expectativa (Focus)
    fig.add_trace(
        go.Scatter(
            x=df["reuniao_data"],
            y=df[col_exp],
            mode="lines+markers",
            name=f"Expectativa ({janela_selecionada})",
            line=dict(color="#636EFA", width=2, dash="dash"),
            hovertemplate="Data: %{x}<br>Expectativa: %{y:.2f}%<extra></extra>",
        )
    )

    # Estilização
    fig.update_layout(
        title={
            "text": f"<b>Focus vs COPOM (2022+) — {janela_selecionada}</b><br><sup>MAE (Erro Médio Absoluto): {mae:.2f} p.p. | Viés Médio: {bias:.2f} p.p.</sup>",
            "x": 0.05,
            "xanchor": "left",
        },
        xaxis_title="Data da Reunião do COPOM",
        yaxis_title="Taxa Selic (% a.a.)",
        template="plotly_white",
        hovermode="x unified",
        legend=dict(
            orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1
        ),
        margin=dict(l=40, r=40, t=80, b=40),
        height=500,
    )

    fig.show()

### 4. Renderização

In [5]:
toggle_janela = widgets.ToggleButtons(
    options=list(opcoes_janela.keys()),
    description="Janela:",
    button_style="",
)

interactive_plot = widgets.interactive_output(
    plotar_grafico, {"janela_selecionada": toggle_janela}
)

display(toggle_janela, interactive_plot)

ToggleButtons(description='Janela:', options=('1 Mês Antes', '3 Meses Antes', '6 Meses Antes', '12 Meses Antes…

Output()